# MindLens Distortion Classifier Training

Kaggle T4 x2 notebook for training the MindLens cognitive distortion classifier.


In [ ]:
!pip install -q transformers datasets accelerate huggingface_hub scikit-learn

In [ ]:
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged into HuggingFace")

In [ ]:
import json
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from datasets import load_from_disk
from sklearn.metrics import f1_score, precision_recall_fscore_support, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

LABELS = [
    "catastrophizing",
    "mind_reading",
    "all_or_nothing",
    "personalization",
    "overgeneralization",
    "emotional_reasoning",
    "should_statements",
    "jumping_to_conclusions",
    "magnification",
    "mental_filter",
]

NUM_LABELS = len(LABELS)
MODEL_NAME = "roberta-base"
YOUR_HF_USERNAME = "AmiruMallawarachchi"
OUTPUT_DIR = Path("/kaggle/working/mindlens-distortion-classifier")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kept separate from OUTPUT_DIR on purpose: TrainingArguments writes a full
# checkpoint (model + optimizer state) here every epoch. A previous run
# pointed both at the same folder, so upload_folder(OUTPUT_DIR) swept up
# every intermediate checkpoint alongside the final model and pushed 6.48GB
# to the Hub for what should be a ~500MB RoBERTa-base classifier.
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle's actual mount slug under /kaggle/input/ does not always match the
# name typed at upload time or the URL slug shown in the browser -- both
# have been wrong here before. Discover it instead of hardcoding it: this
# notebook attaches exactly one dataset, so whatever single directory shows
# up under /kaggle/input/ is it.
_input_root = Path("/kaggle/input")
_mounted = [d for d in _input_root.iterdir() if d.is_dir()] if _input_root.exists() else []
print(f"Mounted under /kaggle/input/: {[d.name for d in _mounted]}")

if len(_mounted) == 1:
    DATA_PATH = _mounted[0]
elif len(_mounted) > 1:
    _matches = [d for d in _mounted if "counselchat" in d.name.lower()]
    if len(_matches) == 1:
        DATA_PATH = _matches[0]
    else:
        raise FileNotFoundError(
            f"Multiple datasets mounted and none uniquely match 'counselchat': "
            f"{[d.name for d in _mounted]}. Set DATA_PATH manually to the right one."
        )
else:
    raise FileNotFoundError(
        "Nothing mounted under /kaggle/input/ -- attach the "
        "counselchat-distortion-cleaned dataset via the Input panel first."
    )

print(f"Using DATA_PATH: {DATA_PATH}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
class DistortionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

def extract_labels(example: dict[str, Any]) -> list[int]:
    if "labels" in example and isinstance(example["labels"], list):
        raw = example["labels"]
        if len(raw) == NUM_LABELS:
            return [int(x) for x in raw]
    if "cognitive_distortions" in example:
        vector = [0] * NUM_LABELS
        distortions = example["cognitive_distortions"]
        if isinstance(distortions, list):
            for item in distortions:
                label = str(item).strip().lower()
                if label in LABELS:
                    vector[LABELS.index(label)] = 1
        return vector
    raise ValueError(f"Cannot extract labels from example: {example}")

class MultiLabelTrainer(Trainer):
    """BCEWithLogitsLoss, optionally weighted per-label via pos_weight.

    A prior run with unweighted loss collapsed to F1 0.0 by epoch 2: on 690
    training examples spread across 10 classes (several with only a handful
    of positive examples), always predicting "no distortion" is nearly
    always correct, so the loss goes down while the model learns nothing.
    pos_weight = n_negative / n_positive per label counteracts that.
    """

    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if not isinstance(labels, torch.Tensor):
            labels = torch.tensor(labels, dtype=torch.float32, device=logits.device)
        else:
            labels = labels.to(logits.device).float()
        weight = self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = 1 / (1 + np.exp(-predictions))
    preds = (probs > 0.5).astype(int)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
# Locate the actual dataset_dict.json + train/ pair under DATA_PATH,
# wherever it really is. Printing the full listing first (rather than only
# a top-level check) means if this still fails, the traceback shows exactly
# what Kaggle actually mounted instead of another guess.
import zipfile

def _list_recursive(folder, depth=2, prefix=""):
    if depth < 0:
        return
    try:
        entries = sorted(folder.iterdir())
    except Exception as exc:
        print(f"{prefix}{folder} -- could not list: {exc}")
        return
    for entry in entries:
        print(f"{prefix}{entry.name}{'/' if entry.is_dir() else ''}")
        if entry.is_dir():
            _list_recursive(entry, depth - 1, prefix + "  ")

print(f"Raw DATA_PATH: {DATA_PATH}")
print("Full listing:")
_list_recursive(DATA_PATH)


def _has_dataset(folder):
    return (folder / "dataset_dict.json").exists() and (folder / "train").is_dir()


def _find_dataset(root, max_depth=3):
    """Search root and its subfolders (up to max_depth) for the
    dataset_dict.json + train/ pair, in case Kaggle nested it."""
    if _has_dataset(root):
        return root
    if max_depth <= 0:
        return None
    for entry in root.iterdir():
        if entry.is_dir():
            found = _find_dataset(entry, max_depth - 1)
            if found:
                return found
    return None


resolved = _find_dataset(DATA_PATH)

if resolved is None:
    # Nothing extracted anywhere under DATA_PATH -- look for a zip and
    # extract it ourselves.
    zips = list(DATA_PATH.rglob("*.zip"))
    if zips:
        extract_to = Path("/kaggle/working/counselchat-extracted")
        extract_to.mkdir(parents=True, exist_ok=True)
        for z in zips:
            print(f"Extracting {z} -> {extract_to}")
            with zipfile.ZipFile(z) as archive:
                archive.extractall(extract_to)
        resolved = _find_dataset(extract_to)

if resolved is None:
    raise FileNotFoundError(
        f"Could not find dataset_dict.json + train/ anywhere under {DATA_PATH} "
        "(listing printed above). Check the dataset actually contains those "
        "files -- open its Data Card on Kaggle and confirm dataset_dict.json "
        "is listed as a real file, not just previewed inside an unextracted zip."
    )

DATA_PATH = resolved
print(f"Resolved DATA_PATH: {DATA_PATH}")
print(f"Contents: {[p.name for p in DATA_PATH.iterdir()]}")

print(f"Loading dataset from: {DATA_PATH}")
ds = load_from_disk(str(DATA_PATH))
train_split = ds["train"]
val_split = ds["validation"] if "validation" in ds else ds["test"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_texts = [ex["text"] for ex in train_split]
train_labels = [extract_labels(ex) for ex in train_split]
val_texts = [ex["text"] for ex in val_split]
val_labels = [extract_labels(ex) for ex in val_split]

train_dataset = DistortionDataset(train_texts, train_labels, tokenizer)
val_dataset = DistortionDataset(val_texts, val_labels, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(LABELS)},
    label2id={label: i for i, label in enumerate(LABELS)},
)

# Per-label positive counts, printed plainly so a class with little or no
# real signal (and therefore little chance of being learned no matter how
# it's weighted) is visible rather than discovered later as an unexplained
# gap on the model card.
train_labels_matrix = np.array(train_labels, dtype=np.float32)
n_pos = train_labels_matrix.sum(axis=0)
n_total = train_labels_matrix.shape[0]
print("Per-label positive examples (of", n_total, "training rows):")
for name, count in sorted(zip(LABELS, n_pos), key=lambda x: -x[1]):
    flag = "  <-- little/no signal" if count < 5 else ""
    print(f"  {name:25s}: {int(count):4d}{flag}")

pos_weight = np.clip((n_total - n_pos) / np.maximum(n_pos, 1), 1.0, 50.0)
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)
print("pos_weight range:", f"{pos_weight.min():.2f} - {pos_weight.max():.2f}")

In [ ]:
args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer = MultiLabelTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    pos_weight=pos_weight_tensor,
)

print("\n>>> STARTING DISTORTION TRAINING <<<\n")
trainer.train()

metrics = trainer.evaluate()
print("\n" + "=" * 60)
print(f"FINAL MACRO F1: {metrics['eval_f1_macro']:.4f}")
print(f"FINAL MICRO F1: {metrics['eval_f1_micro']:.4f}")
print(f"FINAL RECALL:   {metrics['eval_recall_macro']:.4f}")
print(f"FINAL PRECISION:{metrics['eval_precision_macro']:.4f}")
print("=" * 60)

# Macro-F1 blends all 10 classes equally, so a couple of near-zero-support
# classes can drag the average down while well-supported ones do much
# better. Break it out per label rather than reporting one blended number
# that hides which classes actually work.
import pandas as pd

pred = trainer.predict(val_dataset)
probs = 1 / (1 + np.exp(-pred.predictions))
preds = (probs > 0.5).astype(int)
gold = np.array(pred.label_ids)

p, r, f, support = precision_recall_fscore_support(
    gold, preds, average=None, zero_division=0, labels=list(range(NUM_LABELS))
)
per_label = pd.DataFrame({
    "label": LABELS,
    "support": support.astype(int),
    "precision": p.round(3),
    "recall": r.round(3),
    "f1": f.round(3),
}).sort_values("f1", ascending=False)
print("\nPer-label breakdown (validation set):")
print(per_label.to_string(index=False))

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

with open(OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({
        "labels": LABELS,
        "id2label": {i: label for i, label in enumerate(LABELS)},
        "label2id": {label: i for i, label in enumerate(LABELS)},
    }, f, indent=2)

api = HfApi()
repo_id = f"{YOUR_HF_USERNAME}/mindlens-distortion-classifier"
api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_folder(folder_path=str(OUTPUT_DIR), repo_id=repo_id)
print(f"\n>>> UPLOADED: https://huggingface.co/{repo_id}")